# Thực hành Bài kiểm tra 2 – ViHSD

Notebook mặc định chạy Development: Train, CV và Validation. Nó không tải, đọc nhãn, dự đoán hay chấm Test. Có thể bật `VIHSD_NOTEBOOK_USE_FIXTURE=1` để làm bài trên fixture kỹ thuật; fixture không phải kết quả nghiên cứu. Không hiển thị bình luận.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from vihsd.experiment import load_official_frames, DATASET_REVISION, DEVELOPMENT_SPLITS
from vihsd.models import LABEL_IDS, LABEL_NAMES, build_text_pipeline, make_estimators
from vihsd.model_evaluation import (
    classification_metrics, confusion_matrix_rows, cross_validate_text_models,
    duplicate_group_ids, duplicate_group_summary, make_cv_schemes,
)

USE_FIXTURE = os.environ.get('VIHSD_NOTEBOOK_USE_FIXTURE') == '1'
if USE_FIXTURE:
    print('FIXTURE KỸ THUẬT — KHÔNG PHẢI KẾT QUẢ ViHSD')
    train_texts = ['sạch ví dụ a', 'sạch ví dụ b', 'nội dung sạch c', 'bình luận sạch d', 'văn bản sạch e', 'mẫu sạch f',
                   'xúc phạm ví dụ a', 'xúc phạm ví dụ b', 'câu xúc phạm c', 'mẫu xúc phạm d', 'nội dung xúc phạm e', 'văn bản xúc phạm f',
                   'thù ghét ví dụ a', 'thù ghét ví dụ b', 'câu thù ghét c', 'mẫu thù ghét d', 'nội dung thù ghét e', 'văn bản thù ghét f']
    train_y = np.repeat([0, 1, 2], 6)
    validation_texts = ['sạch kiểm tra', 'sạch thêm', 'câu sạch', 'xúc phạm kiểm tra', 'xúc phạm thêm', 'câu xúc phạm', 'thù ghét kiểm tra', 'thù ghét thêm', 'câu thù ghét']
    validation_y = np.repeat([0, 1, 2], 3)
    revision = 'fixture-only'
else:
    frames, revision = load_official_frames(
        PROJECT_ROOT / 'data' / 'raw' / 'uitnlp_vihsd',
        PROJECT_ROOT / 'data' / 'raw' / 'uitnlp_vihsd_provenance.json',
        splits=DEVELOPMENT_SPLITS,
    )
    train_texts = frames['train']['free_text'].tolist()
    train_y = frames['train']['label_id'].to_numpy()
    validation_texts = frames['validation']['free_text'].tolist()
    validation_y = frames['validation']['label_id'].to_numpy()
print('revision:', revision)
print('Development rows:', {'train': len(train_y), 'validation': len(validation_y)})


## 1. Kiểm tra nhãn và baseline
Chỉ xem số lượng nhãn tổng hợp. DummyClassifier tạo baseline thực nghiệm từ Train; Macro F1 của nó không phải con số tham chiếu lý thuyết ở Tuần 4.

In [ ]:
def label_counts(labels):
    values, counts = np.unique(labels, return_counts=True)
    return {LABEL_NAMES[int(label)]: int(count) for label, count in zip(values, counts)}

print('Train:', label_counts(train_y))
print('Validation:', label_counts(validation_y))

estimators = make_estimators(random_state=42)
baseline = build_text_pipeline(estimators['dummy_most_frequent'], feature_mode='word')
baseline.fit(train_texts, train_y)
baseline_validation_pred = baseline.predict(validation_texts)
baseline_metrics = classification_metrics(validation_y, baseline_validation_pred)
pd.DataFrame([{'model': 'dummy_most_frequent', **baseline_metrics}])


## 2. So sánh các classifier trên Validation
Mỗi pipeline được fit chỉ trên Train. Cấu hình giữ cố định; chưa tìm kiếm tham số. Đọc Macro F1 cùng chỉ số riêng của từng lớp.

In [ ]:
validation_rows = []
validation_confusions = []
fitted_pipelines = {}
for model_name, estimator in estimators.items():
    pipeline = build_text_pipeline(estimator, feature_mode='word')
    pipeline.fit(train_texts, train_y)
    train_pred = pipeline.predict(train_texts)
    valid_pred = pipeline.predict(validation_texts)
    row = {'model': model_name}
    row.update({f'train_{k}': v for k, v in classification_metrics(train_y, train_pred).items()})
    row.update({f'validation_{k}': v for k, v in classification_metrics(validation_y, valid_pred).items()})
    validation_rows.append(row)
    validation_confusions.extend(confusion_matrix_rows(validation_y, valid_pred, model_name, 'validation'))
    fitted_pipelines[model_name] = pipeline

validation_results = pd.DataFrame(validation_rows).sort_values('validation_macro_f1', ascending=False)
validation_results[['model', 'train_macro_f1', 'validation_macro_f1', 'validation_f1_clean', 'validation_f1_offensive', 'validation_f1_hate']]

## 3. Cross-validation trên Train
Vectorizer được fit lại trong train của từng fold. StratifiedKFold giữ tỷ lệ lớp; StratifiedGroupKFold đồng thời gom các văn bản trùng theo khóa chuẩn hóa. Hai kết quả trả lời câu hỏi đánh giá khác nhau.

In [ ]:
groups = duplicate_group_ids(train_texts)
group_summary = duplicate_group_summary(train_texts, train_y)
print('Số nhóm văn bản và xung đột nhãn:', group_summary)

cv_rows = []
for scheme_name, splitter in make_cv_schemes(n_splits=3, random_state=42).items():
    cv_rows.append(cross_validate_text_models(
        train_texts, train_y, estimators, splitter, scheme_name,
        feature_mode='word', groups=groups,
    ))
cv_folds = pd.concat(cv_rows, ignore_index=True)
cv_folds.groupby(['scheme', 'model'])['validation_macro_f1'].agg(['mean', 'std']).reset_index()

## 4. Đọc lỗi trên Validation

Chọn mô hình để phân tích theo Validation Macro F1. Không có cell nào của Notebook truy cập Test. Nếu cần đánh giá cuối kỳ, đó là quy trình riêng với cấu hình đã khóa trước và không nằm trong bài thực hành này.


In [ ]:
selected_model = str(validation_results.iloc[0]['model'])
selected_pipeline = fitted_pipelines[selected_model]
print('Đã khóa theo Validation:', selected_model)
validation_confusion = pd.DataFrame(validation_confusions)
validation_confusion[
    (validation_confusion['model'] == selected_model)
    & (validation_confusion['true_label'] != validation_confusion['predicted_label'])
].sort_values('count', ascending=False)

## 5. Bài thực hành cá nhân 10–15 phút

Làm trên fixture hoặc dữ liệu Development (Train/CV/Validation). Không tải hay mở nhãn Test.

- **Tú:** đổi `alpha` của `multinomial_nb_numpy` từ `1.0` thành `0.5`, chạy lại bước so sánh Validation; giải thích baseline luôn đoán CLEAN và đọc một ô ngoài đường chéo của confusion matrix Validation.
- **Tài:** đổi `C` hoặc `class_weight` của Logistic Regression/Linear SVC, chạy lại Validation và CV trên Train; giải thích CV mean/std và vì sao không chọn cấu hình theo Test.

Ghi cấu hình, kết quả tổng hợp và giải thích của chính mình; không đánh dấu hoàn thành thay cho người học.


In [ ]:
# Hàm hỗ trợ kiểm tra đặc trưng cục bộ. Không gọi hoặc lưu bảng chứa từ/cụm từ vào repository.
def local_top_features(fitted_pipeline, class_name, limit=15):
    """Tra ve dac trung lien quan lop de xem trong phien cuc bo, khong xuat file."""
    classifier = fitted_pipeline.named_steps['classifier']
    vectorizer = fitted_pipeline.named_steps['tfidf']
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    class_id = next(label_id for label_id, name in LABEL_NAMES.items() if name == class_name)
    class_position = list(classifier.classes_).index(class_id)
    if hasattr(classifier, 'coef_'):
        weights = np.asarray(classifier.coef_)
    elif hasattr(classifier, 'feature_log_prob_'):
        weights = np.asarray(classifier.feature_log_prob_)
    else:
        raise TypeError('Classifier khong cong bo trong so dac trung')
    if weights.shape[0] != len(classifier.classes_):
        raise ValueError('Khong the gan hang trong so cho tung lop')
    row = weights[class_position]
    indices = np.argsort(row)[-limit:][::-1]
    return pd.DataFrame({'feature': feature_names[indices], 'weight': row[indices]})
